# KM Momentum — 12M — Equal Weight

One signal, one formation horizon and one maintenance method. The experiment contains nine cells: rebalance every 1, 3 or 6 months × target N=12, 24 or 50.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from momentum_india.notebook_views import ResearchNotebook

research = ResearchNotebook('km_momentum', '12M', 'equal_weight')

## 1. Signal and portfolio rule

Quality/liquidity universe → Close ≥ 1.03 × EMA(100) and bullish Supertrend(10,3) → raw-momentum ranking → highest N. There is no secondary proximity or risk ranking. A daily close below EMA(100), or a bearish Supertrend, triggers a sale at the next actual observed open. The proceeds earn the cash-sleeve return until the next scheduled rebalance.

### Technical definition

EMA(100) begins at the mean of the first 100 observed closes and thereafter uses α=2/101. ATR(10) begins at the mean of the first ten true ranges having a prior close, then uses Wilder's α=1/10 smoothing. Supertrend bands are HL2 ± 3×ATR; the prior bands govern direction changes and the active band cannot retreat while direction is unchanged. Direction starts bullish at the first valid ATR bar. Missing observations do not create synthetic indicator bars.

These initialization rules make the calculation reproducible. They do not claim exact equality with an unidentified historical charting-package version. All indicators use split/bonus-adjusted OHLC. The 3% buffer applies only to entry; the EMA exit has no 3% buffer.

Each selected stock targets 1/N of pre-trade equity at a scheduled rebalance. If K<N qualify, target cash is (N−K)/N before charges; eligible stocks are not enlarged to 1/K. Sales precede purchases, which are reduced proportionally if charges or unfilled sales restrict available cash. Actual weights can therefore differ slightly from targets. Membership is rebuilt from current ranks without a retention buffer.

## 2. Universe → ranking → actual portfolio

The example uses the latest monthly N=24 signal and exposes the ranking inputs and actual target weights.

In [2]:
research.snapshot()

Symbol,Formation price return,MDTV (INR)
NSE:CUPID-EQ,664.2%,"2,261,224,535.95"
NSE:ATHERENERG-EQ,259.7%,"2,745,338,022.88"
NSE:KMEW-EQ,196.2%,"175,654,741.20"
NSE:RPTECH-EQ,190.7%,"78,147,051.55"
NSE:SILVERTUC-EQ,157.3%,"65,229,764.81"
NSE:SANSERA-EQ,150.7%,"511,549,310.60"
NSE:NGLFINE-EQ,148.1%,"21,228,409.00"
NSE:SBC-EQ,140.1%,"370,044,564.64"
NSE:PAISALO-EQ,129.1%,"406,661,198.67"
NSE:SKYGOLD-EQ,122.8%,"347,630,026.68"


Symbol,Leg,Actual weight,Current selection,Execution status,Formation price return,MDTV (INR)
NSE:CUPID-EQ,long,4.1%,True,selected,664.2%,"2,261,224,535.95"
NSE:ATHERENERG-EQ,long,4.2%,True,selected,259.7%,"2,745,338,022.88"
NSE:KMEW-EQ,long,4.0%,True,selected,196.2%,"175,654,741.20"
NSE:RPTECH-EQ,long,4.0%,True,selected,190.7%,"78,147,051.55"
NSE:SILVERTUC-EQ,long,4.0%,True,selected,157.3%,"65,229,764.81"
NSE:SANSERA-EQ,long,4.2%,True,selected,150.7%,"511,549,310.60"
NSE:NGLFINE-EQ,long,3.2%,True,selected,148.1%,"21,228,409.00"
NSE:SBC-EQ,long,4.2%,True,selected,140.1%,"370,044,564.64"
NSE:PAISALO-EQ,long,3.2%,True,selected,129.1%,"406,661,198.67"
NSE:SKYGOLD-EQ,long,3.2%,True,selected,122.8%,"347,630,026.68"


Symbol,Formation start,Start adjusted close,Signal close date,End adjusted close,Formation price return
NSE:CUPID-EQ,2025-07-31,30.18,2026-07-31,230.62,664.2%


## 3. Return layers across all nine cells

Raw is before trading charges. After-cost gross/pre-tax deducts modeled trading charges. The long-only post-tax overlay additionally applies the annual equity-gains ledger. The academic reference instead compares raw and borrowing-adjusted layers.

In [3]:
research.layers_bridge()

Rebalance,First date,Last date,Sessions
1M,2007-05-03,2026-08-28,4771
3M,2007-07-02,2026-08-28,4729
6M,2007-07-02,2026-08-28,4729


Rebalance,N,Raw,After costs,Post-tax overlay
1M,12,15.2%,13.4%,11.7%
1M,24,20.8%,18.9%,16.5%
1M,50,20.8%,19.0%,16.6%
3M,12,20.6%,19.6%,17.3%
3M,24,20.4%,19.5%,17.1%
3M,50,20.2%,19.3%,17.1%
6M,12,8.1%,7.6%,7.0%
6M,24,8.9%,8.4%,7.6%
6M,50,10.0%,9.5%,8.7%


## 4. Risk-adjusted results

Sharpe uses daily excess returns relative to the liquid fund. VaR and expected shortfall are historical monthly 95% loss measures. Partial first/last months are included. Time below prior peak counts days awaiting a new all-time high—not losing days. The initial invested capital is included as the first peak.

In [4]:
research.risk_grid()

Rebalance,N,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,12,11.7%,0.310,-64.1%,9.7%
1M,24,16.5%,0.585,-49.4%,7.9%
1M,50,16.6%,0.635,-44.8%,7.4%
3M,12,17.3%,0.622,-41.7%,6.6%
3M,24,17.1%,0.722,-32.3%,6.0%
3M,50,17.1%,0.797,-27.7%,5.3%
6M,12,7.0%,0.052,-27.7%,5.8%
6M,24,7.6%,0.096,-21.1%,4.2%
6M,50,8.7%,0.212,-22.5%,3.6%


Rebalance,N,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,12,13.0%,95.4%,1687
1M,24,10.5%,92.2%,812
1M,50,9.7%,91.3%,794
3M,12,8.8%,93.2%,706
3M,24,7.7%,91.4%,730
3M,50,7.2%,89.7%,702
6M,12,8.5%,95.8%,689
6M,24,6.9%,94.0%,731
6M,50,6.4%,92.7%,730


### CAGR

In [5]:
research.heatmap('cagr')

### Sharpe ratio

In [6]:
research.heatmap('sharpe')

### Maximum drawdown

In [7]:
research.heatmap('maximum_drawdown')

### Monthly 95% VaR

In [8]:
research.heatmap('monthly_var_95')

## 5. Equity paths and matched risks

Each chart fixes breadth and compares rebalance frequencies. Final-layer curves and the price benchmark start visible; other return layers remain in the selectable legend. Logarithmic axes make early and late periods comparable; the bottom range slider preserves the full history. Curves display weekly observations for readability, while every statistic uses the complete daily series.

### N=12

In [9]:
research.equity(12)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,15.2%,0.469,-60.5%,9.3%
1M,After costs,13.4%,0.390,-61.8%,9.4%
1M,Post-tax overlay,11.7%,0.310,-64.1%,9.7%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,20.6%,0.800,-36.1%,6.5%
3M,After costs,19.6%,0.750,-37.0%,6.6%
3M,Post-tax overlay,17.3%,0.622,-41.7%,6.6%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,8.1%,0.138,-27.1%,5.6%
6M,After costs,7.6%,0.100,-27.7%,5.8%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,12.5%,94.4%,1643
1M,After costs,12.6%,94.8%,1679
3M,Raw,8.6%,92.1%,658
3M,After costs,8.8%,92.5%,693
6M,Raw,8.4%,95.3%,662
6M,After costs,8.5%,95.5%,689
1M,Post-tax overlay,13.0%,95.4%,1687
3M,Post-tax overlay,8.8%,93.2%,706
6M,Post-tax overlay,8.5%,95.8%,689
1M,Nifty 50 price,13.1%,92.4%,1520


### N=24

In [10]:
research.equity(24)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,20.8%,0.806,-46.0%,7.4%
1M,After costs,18.9%,0.718,-47.4%,7.5%
1M,Post-tax overlay,16.5%,0.585,-49.4%,7.9%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,20.4%,0.942,-27.0%,5.7%
3M,After costs,19.5%,0.883,-28.0%,5.8%
3M,Post-tax overlay,17.1%,0.722,-32.3%,6.0%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,8.9%,0.215,-20.4%,4.2%
6M,After costs,8.4%,0.168,-21.1%,4.2%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,9.6%,90.5%,779
1M,After costs,9.7%,91.1%,805
3M,Raw,7.5%,89.5%,698
3M,After costs,7.6%,90.0%,702
6M,Raw,6.8%,93.2%,639
6M,After costs,6.9%,93.5%,730
1M,Post-tax overlay,10.5%,92.2%,812
3M,Post-tax overlay,7.7%,91.4%,730
6M,Post-tax overlay,6.9%,94.0%,731
1M,Nifty 50 price,13.1%,92.4%,1520


### N=50

In [11]:
research.equity(50)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,20.8%,0.874,-42.0%,7.2%
1M,After costs,19.0%,0.779,-43.1%,7.3%
1M,Post-tax overlay,16.6%,0.635,-44.8%,7.4%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,20.2%,1.037,-22.9%,5.2%
3M,After costs,19.3%,0.973,-24.0%,5.3%
3M,Post-tax overlay,17.1%,0.797,-27.7%,5.3%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,10.0%,0.355,-21.9%,3.4%
6M,After costs,9.5%,0.301,-22.5%,3.5%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,9.1%,89.2%,757
1M,After costs,9.2%,90.1%,767
3M,Raw,7.0%,87.7%,629
3M,After costs,7.2%,88.3%,638
6M,Raw,6.2%,91.8%,630
6M,After costs,6.3%,92.1%,639
1M,Post-tax overlay,9.7%,91.3%,794
3M,Post-tax overlay,7.2%,89.7%,702
6M,Post-tax overlay,6.4%,92.7%,730
1M,Nifty 50 price,13.1%,92.4%,1520


## 6. Recovery burden

The longest underwater episode is shown by its peak, trough and recovery dates. Unrecovered episodes remain explicitly open.

In [12]:
research.recovery()

Series,Peak,Trough,Recovery,Sessions,Calendar days,Episode loss
1M,2018-01-15,2020-04-03,2021-05-06,812,1206,-49.4%
3M,2018-01-15,2019-10-10,2021-01-04,730,1084,-32.3%
6M,2018-01-15,2019-08-22,2021-01-05,731,1085,-20.4%
Nifty · 1M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 3M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 6M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%


## 7. Portfolio behavior

Scheduled turnover is (buy value + sell value)/(2 × pre-trade equity). Retention compares successive scheduled target name sets. Cash and total charges include the intervening daily path.

In [13]:
research.behavior()

Rebalance,N,Mean names at rebalance,Mean cash weight,Scheduled name retention
1M,12,15.41,17.7%,53.8%
1M,24,32.45,17.3%,58.7%
1M,50,63.34,18.3%,58.5%
3M,12,12.92,45.3%,27.7%
3M,24,27.86,42.9%,36.4%
3M,50,57.10,42.8%,39.7%
6M,12,12.49,65.4%,14.9%
6M,24,26.62,62.4%,21.5%
6M,50,55.56,61.8%,27.4%


Rebalance,N,Mean scheduled turnover,All trading charges (INR),Days with stop sales
1M,12,33.9%,"10,394,108.00",810
1M,24,31.6%,"22,159,735.64",1289
1M,50,30.2%,"21,348,966.32",1946
3M,12,48.3%,"12,998,139.00",563
3M,24,44.4%,"12,690,458.54",942
3M,50,41.7%,"12,060,588.87",1455
6M,12,49.6%,"1,623,276.94",354
6M,24,47.1%,"1,903,502.70",611
6M,50,44.7%,"2,124,741.20",970


## 8. Market-state attribution

This is an observation, not an extra strategy filter. Prior-close Nifty 50 versus SMA(200) defines up/down; 63-session volatility versus its expanding historical median defines high/low volatility. The representative monthly N=24 path is shown with shaded states.

In [14]:
research.regimes()

Market state,Sessions,Mean daily return,Daily volatility,Positive days
Down / High volatility,748,0.03%,0.94%,56.02%
Down / Low volatility,609,-0.06%,1.07%,55.34%
Up / High volatility,720,0.22%,1.25%,62.78%
Up / Low volatility,2694,0.07%,1.06%,57.09%


## 9. Complete portfolio and trade evidence

Separate CSV files retain all scheduled portfolios, actual trades and risk layers for this exact signal/lookback/maintenance combination.

In [15]:
research.portfolio_exports()

Rebalance,N,First rebalance,Last rebalance,Rebalance dates,Holding rows
1M,12,2007-05-03,2026-08-03,232,3576
1M,24,2007-05-03,2026-08-03,232,7528
1M,50,2007-05-03,2026-08-03,232,14696
3M,12,2007-07-02,2026-07-01,77,995
3M,24,2007-07-02,2026-07-01,77,2145
3M,50,2007-07-02,2026-07-01,77,4397
6M,12,2007-07-02,2026-07-01,39,487
6M,24,2007-07-02,2026-07-01,39,1038
6M,50,2007-07-02,2026-07-01,39,2167


## Findings

In [16]:
research.conclusion()

Matched reference: [Raw Momentum — 12M — Equal Weight](04_raw_momentum_12m_equal_weight.ipynb). The comparison holds formation, maintenance, frequency and N fixed.